In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [29]:
# 1. Setup Data
df = pd.read_csv('/content/drive/MyDrive/College/Sem 8/DM/CA2/play_tennis.csv')
train,test = train_test_split(df,test_size=0.1,random_state=42)
display(df)

,Outlook,Temp_Value,Humidity_Value,Wind,PlayTennis
0,Sunny,35,85,Weak,No
1,Sunny,34,90,Strong,No
2,Overcast,33,78,Weak,Yes
3,Rain,28,80,Weak,Yes
4,Rain,22,70,Weak,Yes
5,Rain,21,65,Strong,No
6,Overcast,23,72,Strong,Yes
7,Sunny,29,88,Weak,No
8,Sunny,24,68,Weak,Yes
9,Rain,27,75,Weak,Yes


In [30]:
target = df.columns[-1]
classes = train[target].unique()
features = [ f for f  in train.columns if f!=target]

# identify column types once
cat_cols = [ f for f in features if train[f].dtype == 'O']
num_cols = [ f for f in features if f not in cat_cols]


In [31]:
def get_prediction(row):

  probs = {}

  for c in classes :
    # step A: Filter train data for the current class
    data_c = train[ train[target] == c ]

    # step B: Start with Prior Probability P(Clas)
    p = len(data_c) / len(train)

    #step C: Multiply by likelihood P(feature|Class)
    for f in features:
      val = row[f]
      if f in cat_cols:
        #Categorical : Count occurences
        count  = len(data_c[data_c[f]==val])
        p *= (count/len(data_c))
      else:
        #Numerical : Gaussian PDF
        mu, std = data_c[f].mean(),data_c[f].std()
        exponent = np.exp((-1/2)*(((val-mu)/(std))**2))
        p *= (1/(np.sqrt(2*np.pi*std))) * exponent
    probs[c] = p

  max_prob = float('-inf')
  max_class = ""
  for k,v in probs.items():
    if max_prob < v :
      max_prob = v
      max_class = k
  return max_class


In [32]:
#execution
y_true = test[target].tolist()
y_pred = [get_prediction(row) for _,row in test.iterrows()]

print("Actual:   ", y_true)
print("Predicted:", y_pred)


Actual:    ['Yes', 'Yes']
Predicted: ['Yes', 'Yes']
